# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [105]:
ПРІЗВИЩЕ_ІМЯ = 'Штурин Захарій'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-34'
НОМЕР        = 6       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [106]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 6   ·   Штурин Захарій   ·   ШІ-34

Ваші вісім стовпців:
   1. ПІП
   2. Примітка
   3. Перебіг
   4. Дата розтину
   5. Ліжко днів
   6. Т вище38
   7. МНВ1
   8. Креат2

Ваша пара для аналізу зв’язку: Ліжко днів та Вік

Ваше додаткове питання:
   Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?


In [107]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,ПІП,Примітка,Перебіг,Дата розтину,Ліжко днів,Т вище38,МНВ1,Креат2
0,Федорюк Петро Юрійович,NaN,важкий,2022-03-11,14,4 дні до госп,3.10,92
1,Загул Григорій Сергійович,"отримував актемра, часті носові кровотечі, втрата нюху",важкий,NaT,22,5 до гос,1.22,73
2,Бондара Лариса Павлівна,часта втрата свідомосі в стаціонарі,важкий,NaT,20,10 до гос,1.03,117.2
3,Ватаманюка Уляна Дмитрівна,панкреонекроз,важкий,NaT,20,9 до гос,1.69,105.1
4,Бабійа Лариса Олегівна,NaN,важкий,NaT,27,NaN,1.15,101
5,Ковалика ганна Андріївна,NaN,важкий,2021-12-26,9(5 у реан),NaN,0.90,172
6,Височан Віктор Володимирович,"глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",важкий,NaT,18,7 дні до госп,1.09,97.8
7,Дзюбаа Ірина Іванівна,NaN,серед важ,NaT,10,9 днів до госп,0.94,107


### Допоміжні функції

Їх використовують кілька етапів, тому вони зібрані в одному місці.

* `ПОРЯДОК` — явний порядок категорій порядкової змінної (за абеткою його не задати);
* `_як_число` — бере перше число з кожної клітинки: `'9(5 у реан)'` → 9, `'117,2'` → 117.2, `'немає'` → `NaN`;
* `_в_числа` — приводить стовпець до чисел залежно від його змісту (дата / порядкова / звичайне число).

In [108]:
ПОРЯДОК = {'Перебіг': ['серед важ', 'важкий', 'вкрай важкий']}


def _як_число(з):
    return pd.to_numeric(
        з.astype(str).str.replace(',', '.').str.extract(r'(-?\d+(?:\.\d+)?)')[0],
        errors='coerce')


def _в_числа(стовпець, з):
    if pd.api.types.is_datetime64_any_dtype(з):
        return з
    if стовпець in ПОРЯДОК:
        return з.str.strip().map({v: i for i, v in enumerate(ПОРЯДОК[стовпець])})
    return _як_число(з)

---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [109]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ПІП           121 non-null    str           
 1   Примітка      45 non-null     str           
 2   Перебіг       121 non-null    str           
 3   Дата розтину  61 non-null     datetime64[us]
 4   Ліжко днів    120 non-null    object        
 5   Т вище38      77 non-null     object        
 6   МНВ1          90 non-null     float64       
 7   Креат2        102 non-null    object        
dtypes: datetime64[us](1), float64(1), object(3), str(3)
memory usage: 7.7+ KB


In [110]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- ПІП
    непорожніх 121 з 121 | унікальних 121 | типи: str×121
    приклади: ["'Федорюк Петро Юрійович'", "'Загул Григорій Сергійович'", "'Бондара Лариса Павлівна'", "'Ватаманюка Уляна Дмитрівна'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- Перебіг
    непорожніх 121 з 121 | унікальних 3 | типи: str×121
    приклади: ["'важкий'", "'серед важ'", "'вкрай важкий '"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- Ліжко днів
    непорожніх 120 з 121 | унікальних 40 | типи: int×119, str×1
    приклади: ['14', '22', '20', '27']

--- Т вище38
    непорожніх 77 з 121 | унікаль

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [111]:
ПАСПОРТ = {
    # 'назва стовпця': ('шкала', 'обґрунтування однією фразою'),
    'ПІП':          ('ідентифікатор', 'Прізвище, ім’я та по батькові лише називає людину; з ним немає жодної величини, яку можна порівнювати, окрім рівності.'),
    'Примітка':     ('номінальна', 'Довільний текстовий опис; значення можна лише порівняти на рівність, природного порядку між ними немає.'),
    'Перебіг':      ('порядкова', 'Категорії тяжкості серед важ < важкий < вкрай важкий мають природний порядок, але відстані між рівнями невідомі.'),
    'Дата розтину': ('інтервальна', 'Календарна дата: різниці мають сенс, але природного нуля немає - початок відліку умовний.'),
    'Ліжко днів':   ('відношень', 'Тривалість перебування в днях має природний нуль; відношення на кшталт удвічі довше осмислені.'),
    'Т вище38':     ('відношень', 'Кількість днів із температурою понад 38 має природний нуль; це справжня кількісна величина, а не мітка.'),
    'МНВ1':         ('відношень', 'Кількісний лабораторний показник згортання крові; значення можна порівнювати за величиною, а різниці мають зміст.'), #згортання крові
    'Креат2':       ('відношень', 'Концентрація креатиніну має природний нуль, тому і різниці, і відношення значень осмислені.'), #нирки
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,ПІП,ідентифікатор,"Прізвище, ім’я та по батькові лише називає людину; з ним..."
1,Примітка,номінальна,Довільний текстовий опис; значення можна лише порівняти ...
2,Перебіг,порядкова,Категорії тяжкості серед важ < важкий < вкрай важкий маю...
3,Дата розтину,інтервальна,"Календарна дата: різниці мають сенс, але природного нуля..."
4,Ліжко днів,відношень,Тривалість перебування в днях має природний нуль; віднош...
5,Т вище38,відношень,Кількість днів із температурою понад 38 має природний ну...
6,МНВ1,відношень,Кількісний лабораторний показник згортання крові; значен...
7,Креат2,відношень,"Концентрація креатиніну має природний нуль, тому і різни..."


In [112]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [113]:
def допустима_статистика(стовпець, шкала):
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    частоти = з.value_counts()
    р['мода'] = частоти.index[0]
    р['частота_моди'] = int(частоти.iloc[0])
    if ш == 'номінальна':
        return р

    ч = _в_числа(стовпець, з).dropna()
    р['медіана'] = ч.median()
    р['Q1'] = ч.quantile(0.25)
    р['Q3'] = ч.quantile(0.75)
    if ш == 'порядкова':
        return р

    р['середнє'] = ч.mean()
    р['стандартне_відхилення'] = ч.std()
    if ш == 'інтервальна':
        return р

    р['коефіцієнт_варіації'] = ч.std() / ч.mean()
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- ПІП  (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 121}

--- Примітка  (номінальна)
    {'непорожніх': 45, 'унікальних': 44, 'мода': 'Поліалергетик', 'частота_моди': 2}

--- Перебіг  (порядкова)
    {'непорожніх': 121, 'унікальних': 3, 'мода': 'важкий', 'частота_моди': 90, 'медіана': np.float64(1.0), 'Q1': np.float64(1.0), 'Q3': np.float64(1.0)}

--- Дата розтину  (інтервальна)
    {'непорожніх': 61, 'унікальних': 50, 'мода': Timestamp('2022-06-07 00:00:00'), 'частота_моди': 3, 'медіана': Timestamp('2022-04-23 00:00:00'), 'Q1': Timestamp('2022-03-29 00:00:00'), 'Q3': Timestamp('2022-05-07 00:00:00'), 'середнє': Timestamp('2022-04-06 13:22:37.377049'), 'стандартне_відхилення': Timedelta('72 days 13:34:38.466302')}

--- Ліжко днів  (відношень)
    {'непорожніх': 120, 'унікальних': 40, 'мода': 10, 'частота_моди': 12, 'медіана': np.float64(15.0), 'Q1': np.float64(10.0), 'Q3': np.float64(22.0), 'середнє': np.float64(17.883333333333333), 'стандартне_відхилення': np.float64(12.936

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [114]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ПІП,121,121,Федорюк Петро Юрійович,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Примітка,45,44,Поліалергетик,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Перебіг,121,3,важкий,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Т вище38,77.0,28.0,3.0,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
МНВ1,90.0,NaN,NaN,NaN,1.333778,0.8,1.0525,1.19,1.34,10.6,1.032131
Креат2,102.0,72.0,77.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [115]:
код_перебігу = _в_числа('Перебіг', df['Перебіг'])
середнє_коду = код_перебігу.mean()

наївне = pd.to_numeric(df['Т вище38'], errors='coerce')
коректне = _як_число(df['Т вище38'])

частота_моди_піп = df['ПІП'].value_counts().iloc[0]


In [116]:
БЕЗГЛУЗДІ = [
    ('Перебіг', 'mean від кодів 0/1/2', f'{середнє_коду:.2f}',
     'Коди 0, 1, 2 лише позначають порядок'),
    ('Т вище38', 'mean', f'{наївне.mean():.2f} (n={наївне.notna().sum()}) замість {коректне.mean():.2f} (n={коректне.notna().sum()})',
     f'Через некоректну обробку 9(5 днів до госп) і тд'),
    ('ПІП', 'мода', f'частота моди = {частота_моди_піп} зі {df["ПІП"].notna().sum()}',
     'Усі значення унікальні, тож «найчастіше значення» не існує.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Перебіг,mean від кодів 0/1/2,0.76,"Коди 0, 1, 2 лише позначають порядок"
1,Т вище38,mean,3.44 (n=27) замість 4.22 (n=77),Через некоректну обробку 9(5 днів до госп) і тд
2,ПІП,мода,частота моди = 1 зі 121,"Усі значення унікальні, тож «найчастіше значення» не існує."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [117]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Ліжко днів × Вік

Ліжко днів: шкала — відношень
Вік: шкала — не у вашому списку

    Ліжко днів  Вік
0           14   55
1           22   43
2           20   54
3           20   68
4           27   65
5  9(5 у реан)   66


In [118]:
ОБРАНА_МІРА = 'Кореляція Пірсона'        # напишіть назву міри
ЧОМУ_САМЕ_ВОНА = ('Обидві змінні пари - Ліжко днів і Вік - мають шкалу відношень із природним нулем, тому лінійний коефіцієнт Пірсона для них допустимий')     # одне-два речення

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

from scipy import stats

пара = pd.DataFrame({A: _як_число(df[A]), B: pd.to_numeric(df[B], errors='coerce')}).dropna()
r, _ = stats.pearsonr(пара[A], пара[B])

print(f'Пірсон   r = {r:+.3f}')

Пірсон   r = -0.100


**Висновок до 4.1** Кореляція Пірсона r = -0.1 свідчить про дуже слабкий від’ємний лінійний зв’язок між віком і кількістю днів перебування в лікарні. 

In [119]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
стать = df['Стать'].astype(str).str.strip().str[0]
помер = (df['помер/жив'] == 1).astype(int)

r_1 = стать.map({'ч': 0, 'ж': 1}).corr(помер)
r_2 = стать.map({'ч': 1, 'ж': 0}).corr(помер)

print(f'Пірсон «Стать» × «помер/жив», ч=0 ж=1: r = {r_1:+.3f}')
print(f'Пірсон «Стать» × «помер/жив», ч=1 ж=0: r = {r_2:+.3f}')

Пірсон «Стать» × «помер/жив», ч=0 ж=1: r = +0.177
Пірсон «Стать» × «помер/жив», ч=1 ж=0: r = -0.177


**Висновок до 4.2.** Пірсон між «Стать» і «помер/жив» застосовувати не можна: обидві змінні номінальні,
а коди 0/1 для статей задані довільно, тому їх зміна міняє коефіцієнт

---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [120]:
ПІДГОТОВКА = {
    'ПІП':          'ідентифікатор - у модель не беремо взагалі',
    'Примітка':     'номінальна - one-hot; порожню примітку позначаємо окремою категорією, рідкісні зводимо в «інше»',
    'Перебіг':      'порядкова - впорядковані коди: серед важ=0, важкий=1, вкрай важкий=2',
    'Дата розтину': 'інтервальна - переходимо до тривалості (днів від найранішої дати) і стандартизуємо',
    'Ліжко днів':   'відношень - спершу витягуємо число з тексту, далі стандартизація; через скошеність за потреби log1p',
    'Т вище38':     'відношень - спершу витягуємо число з тексту, далі стандартизація',
    'МНВ1':         'відношень - стандартизація',
    'Креат2':       'відношень - стандартизація; значення «немає» перетворюємо на пропуск',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,ПІП,ідентифікатор,ідентифікатор - у модель не беремо взагалі
1,Примітка,номінальна,номінальна - one-hot; порожню примітку позначаємо окремо...
2,Перебіг,порядкова,"порядкова - впорядковані коди: серед важ=0, важкий=1, вк..."
3,Дата розтину,інтервальна,інтервальна - переходимо до тривалості (днів від найрані...
4,Ліжко днів,відношень,"відношень - спершу витягуємо число з тексту, далі станда..."
5,Т вище38,відношень,"відношень - спершу витягуємо число з тексту, далі станда..."
6,МНВ1,відношень,відношень - стандартизація
7,Креат2,відношень,відношень - стандартизація; значення «немає» перетворюєм...


In [121]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
def стандартизувати(ряд):
    assert ряд.std() > 0, f'{ряд.name}: сталий стовпець - ділення на нуль'
    return (ряд - ряд.mean()) / ряд.std()


примітка = df['Примітка'].fillna('без_примітки').str.strip().str.lower()
думки = pd.get_dummies(примітка, prefix='примітка').astype(int)
часті = думки.columns[думки.sum() > 2]
думки = думки[часті].assign(примітка_інше=думки.drop(columns=часті).any(axis=1).astype(int))

перебіг = pd.Categorical(df['Перебіг'].str.strip(), categories=ПОРЯДОК['Перебіг'], ordered=True)
assert (перебіг.codes >= 0).all(), 'є категорії поза заданим порядком (вони б отримали код -1)'
перебіг_код = pd.Series(перебіг.codes, index=df.index, name='Перебіг_код')

дати = pd.to_datetime(df['Дата розтину'], errors='coerce')
дата_стд = стандартизувати((дати - дати.min()).dt.days.rename('Дата_розтину_стд'))

ліжко_стд = стандартизувати(_як_число(df['Ліжко днів']).rename('Ліжко_днів_стд'))

підготовлені = pd.concat([думки, перебіг_код, дата_стд, ліжко_стд], axis=1)

print('Готових ознак:', підготовлені.shape[1])
print(підготовлені.round(2).head(6).to_string())

Готових ознак: 5
   примітка_без_примітки  примітка_інше  Перебіг_код  Дата_розтину_стд  Ліжко_днів_стд
0                      1              0            1             -0.37           -0.30
1                      0              1            1               NaN            0.32
2                      0              1            1               NaN            0.16
3                      0              1            1               NaN            0.16
4                      1              0            1               NaN            0.70
5                      1              0            1             -1.40           -0.69


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [122]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [123]:
# Те саме, але вже після перетворення тексту на числа (перевірка вище бачить лише ті значення, що й так числа)
сталі = []
for c in МОЇ:
    if ПАСПОРТ[c][0] in ('інтервальна', 'відношень', 'абсолютна'):
        if _в_числа(c, df[c].dropna()).dropna().nunique() <= 1:
            сталі.append(c)

print('Сталі стовпці серед моїх:', сталі if сталі else 'немає - нормалізація безпечна.')

Сталі стовпці серед моїх: немає - нормалізація безпечна.


In [124]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [125]:
ВИПРАВЛЕННЯ = [
    (1, 'середнє номінальної змінної «помер/жив»',
     'Значення 1 і 2 - це лише мітки (помер/вижив), а не числа; середнє 1.50 не відповідає жодному пацієнтові, а «посередині» - вигадка кодування.',
     'рахувати частоти та частки; мода замість середнього'),

    (2, '«Стать» має 4 значення, але це та сама величина у двох кодуваннях',
     'nunique() дає 4, бо «ж»/«жін» і «ч»/«чол» - це дублікати одного поняття; тверджень «4 статі» не існує, групи штучно роздвоєно.',
     'уніфікувати текст до двох категорій: ж = 75, ч = 46, і далі працювати з ними'),

    (3, 'середнє порядкової «ступінь КТ», закодованої за абеткою',
     'sort_index() сортує назви абеткою, тож коди довільні; середнє 5.9 для порядкової шкали не має сенсу - не можна ділити відстані між ступенями. До того ж категорії містять дублікати та помилки («легка», «сердній», «серед ступ»).',
     'спершу уніфікувати написання, задати порядок тяжкості (легкий < середній < важкий < вкрай важкий < критичний) і брати медіану та моду'),

    (4, 'середнє/медіана «КТ» після to_numeric(errors="coerce")',
     '17 із 44 значень - це діапазони, які тихо перетворились на NaN; 39.6% порахували лише на 27 випадках, втративши третину.',
     'спершу розпарсити діапазони в середину (65-70 → 67.5%) і однакові одиниці, і лише потім рахувати'),

    (5, 'середнє й мінімум «spO2 без дот» разом зі значенням 0.9',
     '0.9 - це частка (0.9 = 90%), а решта - відсотки; змішані кодування дають фальшивий min = 0.9% і зсувають середнє 82.05.',
     'привести 0.9 до 90 і перевірити діапазон, а тоді рахувати середнє на чистій шкалі'),

    (6, 'середня дата та коефіцієнт варіації дат госпіталізації',
     'дата - інтервальна шкала без природного нуля, а CV вимагає шкали відношень: відлік від 1970 року довільний, тож «0.6% однорідності» не існує.',
     'аналізувати тривалості (ліжко-дні, дні до поступлення), а не самі дати; замість CV - розкид дат від початку до кінця'),

    (7, 'кореляція Пірсона між «Стать» та результатом лікування',
     'обидві змінні номінальні; числа 0/1 на статтях довільні, тож r = 0.177 неінтерпретований - це властивість коду, а не зв’язку.',
     'для номінальних брати x^2 та V Крамера: x^2 = 3.09, p ≈ 0.08, V = 0.16 (слабкий, на межі значущості)'),

    (8, 'нормалізація (x - min)/(max - min) усіх числових стовпців без перевірки',
     'сталий стовпець «ФЕРИТ2» (єдине значення 1265) дає ділення на нуль і весь стовпець NaN; крім того, у нормалізацію потрапили ціль («помер/жив»).',
     'виключити ціль і сталі стовпці (перевірка min == max або nunique ≤ 1) перед нормалізацією'),

    (9, 'df.dropna() як «очищення» набору',
     'після видалення рядків із будь-яким пропуском не лишилось жодного рядка - 0 із 121; «повністю заповнений набір» виявився порожнім.',
     'заповнювати пропуски або видаляти стовпці, а не рядки: dropna(subset=[...]) із обґрунтованим списком'),

    (10, 'рейтинг важливості ознак через кореляцію; «Дата розтину» r = 1.00',
     'дата розтину є лише в померлих (61 із 61) і відсутня в живих, тож r = 1.00 - це витік цільової ознаки, а не предиктор. Окрім того, рейтинг не враховує кількість спостережень: «ІЛ6-2» (r = 0.62) пораховано лише на 6 пацієнтах.',
     'виключити післярезультатні ознаки (дата розтину) і ціль, ігнорувати ознаки з малою кількістю значень (n < 30), оцінювати важливість на ознаках, відомих до результату'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,середнє номінальної змінної «помер/жив»,"Значення 1 і 2 - це лише мітки (помер/вижив), а не числа...",рахувати частоти та частки; мода замість середнього
1,2,"«Стать» має 4 значення, але це та сама величина у двох к...","nunique() дає 4, бо «ж»/«жін» і «ч»/«чол» - це дублікати...","уніфікувати текст до двох категорій: ж = 75, ч = 46, і д..."
2,3,"середнє порядкової «ступінь КТ», закодованої за абеткою","sort_index() сортує назви абеткою, тож коди довільні; се...","спершу уніфікувати написання, задати порядок тяжкості (л..."
3,4,"середнє/медіана «КТ» після to_numeric(errors=""coerce"")","17 із 44 значень - це діапазони, які тихо перетворились ...",спершу розпарсити діапазони в середину (65-70 → 67.5%) і...
4,5,середнє й мінімум «spO2 без дот» разом зі значенням 0.9,"0.9 - це частка (0.9 = 90%), а решта - відсотки; змішані...","привести 0.9 до 90 і перевірити діапазон, а тоді рахуват..."
5,6,середня дата та коефіцієнт варіації дат госпіталізації,"дата - інтервальна шкала без природного нуля, а CV вимаг...","аналізувати тривалості (ліжко-дні, дні до поступлення), ..."
6,7,кореляція Пірсона між «Стать» та результатом лікування,"обидві змінні номінальні; числа 0/1 на статтях довільні,...","для номінальних брати x^2 та V Крамера: x^2 = 3.09, p ≈ ..."
7,8,нормалізація (x - min)/(max - min) усіх числових стовпці...,сталий стовпець «ФЕРИТ2» (єдине значення 1265) дає ділен...,виключити ціль і сталі стовпці (перевірка min == max або...
8,9,df.dropna() як «очищення» набору,після видалення рядків із будь-яким пропуском не лишилос...,"заповнювати пропуски або видаляти стовпці, а не рядки: d..."
9,10,рейтинг важливості ознак через кореляцію; «Дата розтину»...,дата розтину є лише в померлих (61 із 61) і відсутня в ж...,"виключити післярезультатні ознаки (дата розтину) і ціль,..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [126]:
# ваш код: три правильні обчислення замість хибних
from scipy import stats

print('Розділ 1 - середнє =', round(df['помер/жив'].mean(), 2), '(хибно: це мітки, а не числа)')
частки = df['помер/жив'].map({1: 'помер', 2: 'вижив'}).value_counts(normalize=True).round(3)
print('  правильно - частки:', частки.to_dict())

стать = df['Стать'].astype(str).str.strip().str[0]
табл = pd.crosstab(стать, df['помер/жив'])
chi2, p, dof, _ = stats.chi2_contingency(табл)
V = (chi2 / (табл.values.sum() * (min(табл.shape) - 1))) ** 0.5
print()
print('Розділ 7 - Пірсон 0.177 неінтерпретований (обидві шкали номінальні)')
print(f'  правильно - x^2 = {chi2:.2f}, p = {p:.3f}, V Крамера = {V:.3f}')

ціль = (df['помер/жив'] == 1).astype(int)
ознаки = df.select_dtypes('number').drop(columns=['помер/жив'])    # «Дата розтину» — не число, тож сюди не потрапляє
ознаки = ознаки.loc[:, ознаки.notna().sum() >= 30]                 # відсікаємо «ІЛ6-2» (n = 6) та подібні
рейтинг = ознаки.corrwith(ціль).abs().sort_values(ascending=False).head(5)
print()
print('Розділ 10 - ТОП-5 ознак без «Дата розтину» й без ознак із n < 30:')
print(pd.DataFrame({'|r|': рейтинг.round(3), 'n': ознаки.notna().sum()[рейтинг.index]}).to_string())

Розділ 1 - середнє = 1.5 (хибно: це мітки, а не числа)
  правильно - частки: {'помер': 0.504, 'вижив': 0.496}

Розділ 7 - Пірсон 0.177 неінтерпретований (обидві шкали номінальні)
  правильно - x^2 = 3.09, p = 0.079, V Крамера = 0.160

Розділ 10 - ТОП-5 ознак без «Дата розтину» й без ознак із n < 30:
         |r|    n
Лім2   0.645  104
Л2     0.480  104
Вік    0.449  121
П/Я2   0.395  104
АЛАТ1  0.392  115


---
# Етап 7. Висновки та індивідуальне питання

In [127]:
ВИСНОВКИ = [
    'З восьми стовпців мого варіанта лише чотири (Ліжко днів, Т вище38, МНВ1, Креат2) мають шкалу відношень і допускають середнє та коефіцієнт варіації; решта - ідентифікатор, номінальна, порядкова й інтервальна - такі статистики забороняють.',
    'Для пари «Ліжко днів» × «Вік», обидві зі шкалою відношень, Пірсон дає r ≈ -0.10: зв’язок слабкий.',
    'Три мої стовпці містять пастки: у «Ліжко днів» трапляється «9(5 у реан)», «Т вище38» змішує текст і числа, а в «Креат2» слово «немає» тихо випадає при перетворенні - усе це треба прибирати ще до аналізу.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. З восьми стовпців мого варіанта лише чотири (Ліжко днів, Т вище38, МНВ1, Креат2) мають шкалу відношень і допускають середнє та коефіцієнт варіації; решта - ідентифікатор, номінальна, порядкова й інтервальна - такі статистики забороняють.
2. Для пари «Ліжко днів» × «Вік», обидві зі шкалою відношень, Пірсон дає r ≈ -0.10: зв’язок слабкий.
3. Три мої стовпці містять пастки: у «Ліжко днів» трапляється «9(5 у реан)», «Т вище38» змішує текст і числа, а в «Креат2» слово «немає» тихо випадає при перетворенні - усе це треба прибирати ще до аналізу.


In [128]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'У файлі три стовпці-ідентифікатори: «номер і/х», «ПІП» та «Телефон». '
    'Ідентифікатор - це не величина, а лише ключ запису, тож розпізнають його не за назвою, '
    'а за структурою значень: майже кожен рядок має власне, унікальне значення. '
    'Практичне правило: рахуємо відношення nunique/notna разом із кількістю заповнених рядків; '
    'якщо відношення ≥ 0.95 і стовпець заповнений більш ніж на 100 із 121 рядків, '
    'це ідентифікатор. За таким порогом проходять рівно три стовпці: ПІП (121 унікальне з 121), '
    'Телефон (120 з 120) і номер і/х (119 з 121 - два номери повторюються, але це все одно ключ). '
    'Вимога великого покриття потрібна, щоб не сплутати їх із числовими показниками, які теж '
    'унікальні, але заповнені лише у кількох пацієнтів: ІЛ6-2, ФЕРИТ1 та ФЕРИТ2 мають '
    'відношення 1.0, проте лише 6, 2 і 1 значення - на такому дрібному зрізі унікальність нічого не доводить.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Скільки стовпців файлу є ідентифікаторами? Як їх розпізнати автоматично, не дивлячись на назви?

У файлі три стовпці-ідентифікатори: «номер і/х», «ПІП» та «Телефон». Ідентифікатор - це не величина, а лише ключ запису, тож розпізнають його не за назвою, а за структурою значень: майже кожен рядок має власне, унікальне значення. Практичне правило: рахуємо відношення nunique/notna разом із кількістю заповнених рядків; якщо відношення ≥ 0.95 і стовпець заповнений більш ніж на 100 із 121 рядків, це ідентифікатор. За таким порогом проходять рівно три стовпці: ПІП (121 унікальне з 121), Телефон (120 з 120) і номер і/х (119 з 121 - два номери повторюються, але це все одно ключ). Вимога великого покриття потрібна, щоб не сплутати їх із числовими показниками, які теж унікальні, але заповнені лише у кількох пацієнтів: ІЛ6-2, ФЕРИТ1 та ФЕРИТ2 мають відношення 1.0, проте лише 6, 2 і 1 значення - на такому дрібному зрізі унікальність нічого не доводить.


In [129]:
# Автоматичне правило: ідентифікатор майже весь унікальний і майже весь заповнений
покриття = df.notna().sum()
унікальність = df.nunique() / покриття
ідентифікатори = покриття[(покриття > 100) & (унікальність >= 0.95)].index.tolist()
print('Знайдено за правилом:', ідентифікатори)
print()


Знайдено за правилом: ['номер і/х', 'ПІП', 'Телефон']



---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [130]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'синтаксис',
    'етап 3 — безглузді числа':        'ні',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'синтаксис',
    'етап 6 — виправлення аналізу':    'ні',
    'етап 7 — висновки й відповідь':   'ні',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'весь написаний синтаксис, оскільки я не знав багатьох функцій для роботи з таблицями через pandas, також я запитував ші про використані метрики, бо ніколи раніше їх не використовував'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [131]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 6 готова до захисту.
Штурин Захарій, ШІ-34


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.